> **LLM Assistance Disclosure:** This notebook was developed with the assistance of Claude (Anthropic), ChatGPT (OpenAI) and Gemini (Google) as a coding and research support tool. All methodological decisions, analytical choices, variable definitions, causal identification strategies, and interpretations are those of the project team. The analysis, conclusions, and narrative are the team's own work.

# DAI Mission — Proposal
**Data & AI in Economics | TU Dortmund**

- **Team size:** 3 students  
- **Deliverable:** This Jupyter Notebook (proposal → final submission in one file)

## 1. Team

| Role | Name | Student ID |
|------|------|------------|
| Lead | Manav Madhukar Dalvi | *(ID)* |
| Member | Bademba Drammeh | *(ID)* |
| Member | Sasha Marie Stühmer | *(ID)* |

## 2. Mission Title & Research Question

**Title:** Beyond Raw Pace: Strategic Decision-Making and Resource Constraint Management in Formula 1.

---

**Research question:**

To what extent do strategic decisions under uncertainty and resource constraints contribute to competitive success in Formula 1 beyond car performance, and how does the value of strategic behaviour vary across different racing environments?

---

**Why it matters:**

Formula 1 is one of the most data-rich competitive environments in professional sport. While car performance explains much of on-track success, teams also operate under binding resource constraints and must make strategic decisions under uncertainty throughout a season. Engine component allocations are limited by regulation, with additional usage resulting in grid penalties. Safety Car deployments can suddenly reduce the effective cost of a pit stop, creating short-lived strategic opportunities. These situations represent genuine economic decision problems: one concerns the management of scarce resources over a season, while the other involves responding to unexpected changes in incentives during a race. Both have measurable outcomes, direct implications for championship points, and ultimately financial consequences through the Constructors' Championship standings. Studying these decisions provides insight into how teams generate competitive advantages beyond raw car performance.

## 3. Data

### Data Sources

**1. Race Results & Qualifying - Jolpica API (2003–2024)**

- Endpoint: `https://api.jolpi.ca/ergast/f1/{year}/results.json` and `/qualifying.json`
- Access: Python `requests`, paginated. Open, no authentication required.
- Coverage: All race weekends 2003–2024 (~8,966 driver-race observations).

**2. Pit Stop Records - Jolpica API (2012–2024)**
- Endpoint: `https://api.jolpi.ca/ergast/f1/{year}/{round}/pitstops.json`
- Access: Round-by-round queries only - season-level endpoint returns HTTP 400.
- Coverage: 2012–2024 (~10,256 individual pit stop records).

**3. Championship Standings - Jolpica API (2002–2024)**
- Endpoint: `/driverStandings.json` and `/constructorStandings.json`
- Access: One call per season. 2002 included to provide previous-season features for 2003.
- Coverage: Driver and constructor end-of-season standings, 23 seasons.

**4. Circuit, Driver & Constructor Reference Tables - Jolpica API**
- Endpoints: `/circuits.json`, `/drivers.json`, `/constructors.json`
- Access: Single paginated fetch each. Static lookup tables used for joins and the FastF1-to-Ergast driver ID mapping.

**5. Lap-by-Lap Data, Tyre Compounds & Safety Car Messages - FastF1 (2018–2024)**
- Access: `pip install fastf1`; `session.load(laps=True, telemetry=False, messages=True)`
- Coverage: ~114,000 lap observations across 2018–2024. Tyre compound data is only
reliable from 2018 - this is a hard boundary respected throughout all analyses.
- Note: 7 sessions could not be collected due to API availability issues (2018 R14, 2023 R5–R10). These are documented in the limitations. These session will be collected and used in the study later.

**6. Historical Lap Time Aggregates - Jolpica API (2011–2017)**
- Endpoint: `https://api.jolpi.ca/ergast/f1/{year}/{round}/laps.json`
- Access: Round-by-round, aggregate mode only (median, std, fastest lap per driver per race). Full individual-lap mode produces ~800,000 rows and is unnecessary for our analysis.
- Coverage: 2,744 driver-race aggregate rows across 7 seasons.

**7. Race-Day Weather - Open-Meteo Historical API (2003–2024)**
- Endpoint: `https://archive-api.open-meteo.com/v1/archive`
- Access: Free, no API key. One call per (circuit, race date) pair using circuit GPS coordinates from Source 4. Race window approximated as hours 12–17 UTC.
- Coverage: 428 unique race-weekend observations.



**Unit of observation:**

The primary analytical unit is one driver–race observation: one row per driver per Grand Prix weekend, covering season, round, circuit, driver, constructor, qualifying result, race result, and all derived strategic features. The master table contains 8,966 such rows (2003–2024).

A secondary unit used in the causal safety car analysis is the safety car event–driver observation: one row per driver per safety car deployment, used to estimate the causal effect of pitting under the safety car (~1,000–1,100 rows, 2018–2024).

---

**Key variables:**

| Variable | Type | Role | Description |
|----------|------|------|-------------|
| `scored_point` | Binary | Target | 1 if driver finished top 10 and scored points |
| `positionOrder` | Integer 1–20 | Target (secondary) | Final finishing position |
| `quali_gap_pct` | Continuous % | Feature / Confounder | Qualifying time gap to pole as a percentage. Primary car quality proxy - pre-race, continuous, observable |
| `grid_position` | Integer 1–20 | Feature | Race starting position after penalties |
| `gridPenalty` | Binary | Feature / Treatment (Block 3) | 1 if driver started further back than qualified AND set a valid Q1 time |
| `gridPenaltyPlaces` | Integer ≥0 | Feature | Grid places lost to penalty |
| `underSC` | Binary per lap | Instrument (Block 2) | 1 if lap run under full safety car |
| `pittedThisLap` | Binary per lap | Treatment (Block 2) | 1 if driver pitted at end of this lap |
| `pitted_under_sc` | Binary | Treatment (Block 2, race level) | 1 if driver pitted during a safety car window - *derived from `underSC` and `pittedThisLap`* |
| `TyreLife` | Integer laps | Confounder (Block 2) | Laps on current tyre set at SC deployment |
| `Compound` | Categorical | Feature (2018–2024 only) | Tyre compound: SOFT / MEDIUM / HARD / INTERMEDIATE / WET |
| `prevSeasonConstructorPos` | Integer 1–10 | Feature / Confounder | Constructor championship position previous season |
| `constructor_quality_quintile` | Integer 1–5 | Moderator | Within-season quintile of `quali_gap_pct` - *derived from `quali_gap_pct`* |
| `circuit_cluster` | Integer 1–4 | Moderator | Circuit archetype from Block 1 - *output of unsupervised analysis* |
| `penalty_cost_elasticity` | Continuous | Feature / Novel variable | OLS slope of finishing position on grid position per circuit - *computed in Block 3* |
| `rollingAvgPos5` | Continuous | Feature | Driver's mean finishing position over last 5 races - *derived, walk-forward safe* |
| `historical_sc_pit_rate` | Continuous 0–1 | Feature (Block 4) | Team's SC pitting rate in last 5 races with SC events - *derived, walk-forward safe* |
| `totalPitStops` | Integer | Feature (2012–2024) | Total pit stops per race |
| `rain_flag` | Binary | Feature | 1 if any race-window precipitation exceeded 0.5mm |
| `avg_air_temp_c` | Continuous | Feature | Mean air temperature during race hours |
| `grid_to_pos` | Integer | EDA / Derived | Grid minus final position - *intermediate variable, used in Block 1 circuit feature computation* |

---

**Potential data quality issues:**

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

master = pd.read_csv("/content/master_driver_race.csv")

print("=" * 60)
print("DATA QUALITY ASSESSMENT FOR PROPOSAL")
print("=" * 60)

# 1. Missing value rates for key variables
key_vars = [
    "scored_point", "positionOrder", "quali_gap_pct",
    "gridPenalty", "gridPenaltyPlaces", "pitted_under_sc",
    "TyreLife", "Compound", "prevSeasonConstructorPos",
    "constructor_quality_quintile", "rollingAvgPos5",
    "totalPitStops", "rain_flag", "avg_air_temp_c",
]
key_vars_present = [v for v in key_vars if v in master.columns]

print("\n1. MISSING VALUE RATES — KEY VARIABLES")
print(f"{'Variable':<35} {'Missing':>8} {'Rate':>8} {'Scope note'}")
print("-" * 75)
for col in key_vars_present:
    n_miss = master[col].isna().sum()
    rate   = n_miss / len(master)
    # Determine if missingness is structural (expected) or problematic
    if col in ["Compound", "pitted_under_sc", "TyreLife"]:
        note = "structural — FastF1 2018+ only"
    elif col == "totalPitStops":
        note = "structural — pit data 2012+ only"
    elif col == "prevSeasonConstructorPos":
        note = "minor — new constructors / first season"
    elif rate < 0.02:
        note = "negligible"
    elif rate < 0.10:
        note = "acceptable"
    else:
        note = "investigate"
    print(f"  {col:<33} {n_miss:>8,} {rate:>7.1%}  {note}")

# 2. Selection bias check — do penalties cluster in certain teams?
print("\n2. SELECTION BIAS — GRID PENALTIES BY CONSTRUCTOR QUALITY TIER")
if "gridPenalty" in master.columns and "constructor_quality_quintile" in master.columns:
    pen_by_tier = (
        master[master["gridPenalty"] == 1]
        .groupby("constructor_quality_quintile")
        .size()
        .reset_index(name="penalties")
    )
    total_by_tier = (
        master.groupby("constructor_quality_quintile")
        .size()
        .reset_index(name="total")
    )
    bias_check = pen_by_tier.merge(total_by_tier, on="constructor_quality_quintile")
    bias_check["penalty_rate"] = bias_check["penalties"] / bias_check["total"]
    print(bias_check.to_string(index=False))
    print("  Note: unequal rates across tiers indicate selection — "
          "controlled for via quality quintile in Block 3")

# 3. SC event frequency — statistical power check
print("\n3. SC EVENT FREQUENCY — STATISTICAL POWER CHECK")
if "pitted_under_sc" in master.columns:
    sc_races = master[master["pitted_under_sc"].notna()]
    total_races_ff1 = master[master["season"] >= 2018]["season"].count() / 20
    sc_race_count   = sc_races[sc_races["pitted_under_sc"] == 1]["season"].count()
    print(f"  Driver-race obs with FastF1 coverage : {len(sc_races):,}")
    print(f"  Obs where driver pitted under SC     : {sc_race_count:,}")
    print(f"  SC pit rate (2018+ sample)           : "
          f"{sc_race_count/len(sc_races):.1%}")

# 4. Regulatory break check — do key variables shift after 2014?
print("\n4. REGULATORY ERA STABILITY (pre/post 2014 hybrid introduction)")
if "gridPenalty" in master.columns:
    for era, label in [((2003,2013), "Pre-2014"), ((2014,2024), "Post-2014")]:
        sub = master[
            (master["season"] >= era[0]) &
            (master["season"] <= era[1])
        ]
        pen_rate = sub["gridPenalty"].mean()
        sp_rate  = sub["scored_point"].mean()
        print(f"  {label}: penalty rate={pen_rate:.1%}  "
              f"points-scoring rate={sp_rate:.1%}  "
              f"n={len(sub):,}")

# 5. FastF1 session gaps
print("\n5. FASTF1 COVERAGE GAPS")
skipped = [
    "2018 R14 (French GP)",
    "2023 R5 (Miami GP)",
    "2023 R6 (Monaco GP)",
    "2023 R7 (Spanish GP)",
    "2023 R8 (Canadian GP)",
    "2023 R9 (Austrian GP)",
    "2023 R10 (British GP)",
]
print(f"  Sessions not collected due to API issues: {len(skipped)}")
for s in skipped:
    print(f"    - {s}")
total_ff1 = 149
print(f"  Coverage: {total_ff1 - len(skipped)}/{total_ff1} "
      f"({(total_ff1-len(skipped))/total_ff1:.1%})")

DATA QUALITY ASSESSMENT FOR PROPOSAL

1. MISSING VALUE RATES — KEY VARIABLES
Variable                             Missing     Rate Scope note
---------------------------------------------------------------------------
  scored_point                             0    0.0%  negligible
  positionOrder                            0    0.0%  negligible
  quali_gap_pct                          146    1.6%  negligible
  gridPenalty                             24    0.3%  negligible
  gridPenaltyPlaces                       24    0.3%  negligible
  pitted_under_sc                      6,873   76.7%  structural — FastF1 2018+ only
  prevSeasonConstructorPos             1,054   11.8%  minor — new constructors / first season
  constructor_quality_quintile           146    1.6%  negligible
  rollingAvgPos5                         113    1.3%  negligible
  totalPitStops                        3,820   42.6%  structural — pit data 2012+ only
  rain_flag                                0    0.0%  negligi



*   **Structural missingness (expected and handled):**
Three variable groups have high null rates by design, not data error.
Tyre compound and safety car features (`Compound`, `pitted_under_sc`,
`TyreLife`) are null for all pre-2018 rows because FastF1 only covers
2018 onwards - this is a hard boundary respected throughout all
analyses. Pit stop counts (`totalPitStops`) are null for pre-2012 rows
because Ergast pit stop records begin in 2012. These nulls are not
imputed; analyses using these variables are simply scoped to the
relevant subsample.

*   **Grid penalty inference (minor false positive risk):** Grid penalties are not directly recorded in the data and must be inferred
from the mismatch between qualifying position and race grid position.
Drivers who failed to set a qualifying time due to mechanical failure
produce an identical mismatch but are not receiving a penalty. This is
mitigated by requiring a non-null Q1 lap time before flagging a penalty,
removing the majority of false positives. Residual misclassification
rate is estimated to be low given the ~527 penalties identified across
22 seasons align with known reporting.

* **Selection bias in strategic decisions (controlled for):**
Strategic decisions are not randomly assigned. Teams that pit under
safety cars and teams that time grid penalties strategically are
systematically different from those that do not - they tend to be
better-resourced and more experienced. This is the central identification
challenge of the causal blocks, addressed directly through the
instrumental variable design in Block 2 and constructor quality
controls in Block 3.

* **FastF1 session gaps (documented):** Seven sessions (4.7% of the 2018–2024 window) could not be collected
due to persistent API availability issues: 2018 R14 and 2023 R5–R10.
These sessions are excluded from the FastF1-dependent analyses. The
safety car IV sample retains sufficient events for estimation across
the remaining sessions.

* **Regulatory discontinuity (acknowledged):**
The introduction of the hybrid power unit in 2014 significantly changed
the engine component allocation rules, making grid penalties more
frequent and more strategically significant. Pre- and post-2014 periods
are treated as separate eras in Block 3's penalty timing analysis.

## 4. Planned Methods

### 4a. Causal Inference

- [x] Causal graph / DAG (DoWhy)
- [x] Backdoor adjustment
- [x] Instrumental variable
- [x] Propensity score stratification
- [ ] Other: ___

**Justification:**

Teams that pit under safety cars are not randomly selected. Better teams recognise the opportunity faster and also score more points regardless, so a simple comparison would overstate the effect of pitting. Safety Car periods provide a potentially useful source of exogenous variation in pit-stop decisions, although the validity of this assumption requires careful consideration. For the penalty timing question, backdoor adjustment estimates the effect while controlling for team quality and competitive context. A DAG formalises the assumed causal structure for both analyses and guides what needs to be controlled for.

---

### 4b. Supervised Learning

- [x] Logistic regression
- [x] Linear / Ridge / Lasso regression
- [ ] k-Nearest Neighbors
- [ ] Support Vector Machine
- [x] Decision Tree / Random Forest
- [ ] Neural network
- [ ] Other: ___

**Justification:**

The goal is not prediction for its own sake but measuring how much strategic behaviour adds above car quality. Logistic regression establishes this baseline in an interpretable way, with coefficients that directly show each feature's contribution. Random forest then captures non-linear interactions between features that a linear model misses, and its built-in feature importance provides a second lens on which variables drive the prediction. ROC-AUC is used throughout rather than accuracy, since the points-scoring base rate is around 45 percent.

---

### 4c. Unsupervised Learning / Generative Models

- [x] K-Means clustering
- [x] Hierarchical clustering
- [ ] Variational autoencoder
- [ ] GAN
- [x] Other: Principal Component Analysis (PCA) as dimensionality reduction prior to clustering

**Justification:**

Strategic decisions do not have equal value at every circuit. Before testing causal effects, the circuits need to be classified by their strategic character. Five features summarise each circuit across 20 years of race data. PCA removes redundancy between correlated features before K-Means groups circuits into interpretable archetypes. The resulting typology then feeds directly into both causal blocks as a moderating variable, allowing the analysis to test whether strategic effects are larger at some circuit types than others.

## 5. Evaluation Strategy

### Causal Inference
- **Average Treatment Effect (ATE):** Primary output of both causal blocks, estimated via DoWhy.
- **First-stage F-statistic :** Confirms instrument relevance for the safety car IV. Threshold: F > 10.
- **Refutation tests:** Random Common Cause and Placebo Treatment run on all causal models. A robust estimate should be stable under the former and collapse to zero under the latter.
- **Why:** Causal claims require more than a point estimate - refutation tests are the minimum standard for ruling out spurious findings.

### Supervised Learning
- **ROC-AUC (primary):** Correct metric for binary classification with ~45% base rate. Accuracy is not reported - a model predicting "scores points" for every driver achieves 45% accuracy trivially.
- **Precision-Recall AUC (secondary):** Reported for the midfield subsamplewhere class balance is closer to equal and strategic effects are largest.
- **Per-fold AUC standard deviation:** Measures consistency across seasons.A model that is strong in some years and random in others is not useful.
- **Validation design:** Walk-forward validation - train on years ≤ t, teston year t+1. Random cross-validation is excluded as it constitutes dataleakage in a time-series setting.
- **Why this sequence:** AUC gain from adding strategic features above carquality features directly quantifies the predictive contribution of strategy.

### Unsupervised Learning
- **Silhouette Score:** Computed for k = 2 to 6 to select the number of clusters. Measures how well-separated and internally cohesive the circuit groups are.
- **Domain validity:** Cluster assignments are cross-checked against known circuit characteristics. Groupings that contradict established F1 knowledge are treated as a specification warning.
- **Why K-Means:** Low-dimensional feature space (2–3 PCs), roughly spherical expected clusters, and the need to export cluster labels as features for downstream models all favour K-Means over hierarchical methods.

## 6. Work Plan

| Step | Owner | Description |
|------|-------|-------------|
| 1 | Manav | Data collection & cleaning: full pipeline across all sources, master driver-race analytical table |
| 2 | All | EDA: distributions of key variables, qualifying gap by season, safety car frequency by circuit, points-scoring rates by constructor tier |
| 3 | Sasha | Unsupervised block: circuit feature computation, PCA, K-Means and hierarchical clustering, circuit typology and penalty cost elasticity table |
| 4 | Manav | Causal block A: safety car instrumental variable design, DAG construction, DoWhy estimation, heterogeneous treatment effect analysis |
| 5 | Manav | Causal block B: grid penalty timing analysis, per-circuit elasticity computation, strategic timing test, era comparison |
| 6 | Bademba | Supervised block: feature matrix construction, walk-forward validation, model sequence M0–M4, SHAP decomposition |
| 7 | All | Synthesis & write-up: connect findings across all blocks, limitations section, notebook polish |

## 7. Results *(complete for final submission)*

### 7a. Causal Inference

In [ ]:
# Causal inference analysis

### 7b. Supervised Learning

In [ ]:
# Supervised learning analysis

### 7c. Unsupervised / Generative

In [ ]:
# Unsupervised / generative analysis

## 8. Discussion & Conclusion *(complete for final submission)*

Synthesise findings across all three method blocks. What does each lens reveal that the others miss? What are the limitations of your analysis?